In [1]:
#모델을 넣어서 해본다. 흐름을 파악
#데이터 불러오기
from torchvision import datasets
from torchvision.transforms import ToTensor


In [2]:
#학습 데이터 불러오기
train_data=datasets.FashionMNIST(
    root="../Data",
    train=True,
    download=True,
    transform=ToTensor()
)

In [3]:
#테스트 데이터 불러오기
test_data=datasets.FashionMNIST(
    root="../Data",
    train=False,
    download=True,
    transform=ToTensor()
)#여기서 바꿨음

In [4]:
#받은 데이터 확인
print(train_data.data.shape)
print(train_data.targets.shape)

print(test_data.data.shape)
print(test_data.targets.shape)

torch.Size([60000, 28, 28])
torch.Size([60000])
torch.Size([10000, 28, 28])
torch.Size([10000])


In [5]:
#data와 target 분류
train_input=train_data.data
train_target=train_data.targets

test_input=test_data.data
test_target=test_data.targets

In [6]:
#데이터 표준화 및 2차원 행렬
train_scaled=(train_input/255.0).reshape(-1,28,28)
test_scaled=(test_input/255.0).reshape(-1,28,28)

print(train_scaled.shape)
print(test_scaled.shape)

torch.Size([60000, 28, 28])
torch.Size([10000, 28, 28])


---
심층신경망 만들기

In [7]:
import torch
import torch.nn as nn#뉴멀 네트워크?
import torch.optim as optim#optimize
from torch.utils.data import TensorDataset,DataLoader#셔플링

In [8]:
#Train과 Valid
from sklearn.model_selection import train_test_split
train_scaled, val_scaled,train_target,val_target=train_test_split(
    train_scaled,train_target,test_size=0.2,random_state=42
)

In [9]:
#dataset과 dataloader(그대로 사용하지 못함) 생성
batch_size=32#mini batch#묶어서 async로 돌린다.
train_dataset=TensorDataset(train_scaled,train_target)
train_loader=DataLoader(train_dataset,batch_size=batch_size,shuffle=True)#머신러닝 crossvalidate
val_dataset=TensorDataset(val_scaled,val_target)
val_loader=DataLoader(val_dataset,batch_size=batch_size)#검증 셔플 필요없음 학습시에만 에포크, 셔플

---
####모델 정의

In [ ]:
#Model 1: 입력층->은닉층(활성화함수)->출력층
class FashionMNISTModel1(nn.Module):
    def __init__(self):
        super(FashionMNISTModel1, self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(28*28,100)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(100,10)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.softmax(x)
        return x

In [11]:
#Model 2: 입력층->은닉층(활성화함수)->Dropout층:과대적합->출력층
class FashionMNISTModel2(nn.Module):
    def __init__(self):
        super(FashionMNISTModel2, self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(28*28,100)
        self.relu = nn.ReLU()
        #drop
        self.dropout1=nn.Dropout(0.5)
        self.fc2 = nn.Linear(100,10)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout1(x)
        x = self.fc2(x)
        x = self.softmax(x)
        return x

In [12]:
##############
#Model 3: 입력층->은닉층(활성화함수)->dropout층->은닉층(활성화함수)->출력층
class FashionMNISTModel3(nn.Module):
    def __init__(self):
        super(FashionMNISTModel3, self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(28*28,100)
        self.relu = nn.ReLU()
        self.dropout1=nn.Dropout(0.5)
        self.fc2 = nn.Linear(100,50)#은닉층?
        self.relu = nn.ReLU()
        self.fc3 = nn.Linear(50,10)#?????????
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout1(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.fc3(x)
        x = self.softmax(x)
        return x
    ###################

In [13]:
##############
#Model 4: 입력층->은닉층(활성화함수)->dropout층->은닉층(활성화함수)->Dropout->출력층
class FashionMNISTModel4(nn.Module):
    def __init__(self):
        super(FashionMNISTModel4, self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(28*28,100)
        self.relu = nn.ReLU()
        self.dropout1=nn.Dropout(0.5)
        self.fc2 = nn.Linear(100,50)#은닉층?
        self.dropout2=nn.Dropout(0.5)
        self.relu = nn.ReLU()
        self.fc3 = nn.Linear(50,10)#?????????
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout1(x)
        x = self.fc2(x)
        x = self.dropout2(x)
        x = self.relu(x)
        x = self.fc3(x)
        x = self.softmax(x)
        return x
    ###################

In [14]:
device=torch.device("mps" if torch.backends.mps.is_available() else 'cpu')
print(device)
# model.to(device)#summary

mps


In [15]:
#모델, 손실함수, 옵티마이저 초기화
#모델 인스턴스
model1=FashionMNISTModel1().to(device)

#손실함수와 옵티마이저
criterion1=nn.CrossEntropyLoss()
optimizer1=optim.Adam(model1.parameters())

In [ ]:
model2=FashionMNISTModel2().to(device)

#손실함수와 옵티마이저
criterion2=nn.CrossEntropyLoss()
optimizer2=optim.Adam(model2.parameters())

In [17]:
model3=FashionMNISTModel3().to(device)

#손실함수와 옵티마이저
criterion3=nn.CrossEntropyLoss()
optimizer3=optim.Adam(model3.parameters())

In [18]:
model4=FashionMNISTModel4().to(device)

#손실함수와 옵티마이저
criterion4=nn.CrossEntropyLoss()
optimizer4=optim.Adam(model4.parameters())

---
#학습함수

In [19]:
#학습함수(한번 만들고 복사):순서중요
def train(model,train_loader,criterion,optimizer,device):
    model.train()#dropout 포함됨
    total_loss=0
    for inputs,targets in train_loader:
        inputs, targets =inputs.to(device), targets.to(device)#gpu에 옮겨놔야 계산을 해줌<<<<<<<<
        optimizer.zero_grad()#기울기 0:gradient 초기화
        outputs=model(inputs)#정방향
        loss=criterion(outputs,targets)#손실함수
        loss.backward()#역전파
        optimizer.step()#파라미터/가중치 바꿔줌
    return loss.item()

In [20]:
#평가함수
def evaluate(model, val_loader,criterion,device):
    model.eval()#llm:정방향:기울기?미분 바꿈
    total_loss=0#전체 손실 합계
    correct=0#정확하게 예측한 샘플수
    total=0#전체 샘플수
    with torch.no_grad():
        #with file close 안해도됨
        for inputs, targets in val_loader:
            inputs,targets= inputs.to(device),targets.to(device)
            outputs=model(inputs)
            loss=criterion(outputs,targets)
            total_loss+=loss.item()
            _, predicted= outputs.max(1)#선형회귀
            total+=targets.size(0)
            correct+=predicted.eq(targets).sum().item()

    return total_loss/len(val_loader),correct/total #평균손실, 정확도        


In [21]:
model1.to(device)
model2.to(device)
model3.to(device)
model4.to(device)

FashionMNISTModel4(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=784, out_features=100, bias=True)
  (relu): ReLU()
  (dropout1): Dropout(p=0.5, inplace=False)
  (fc2): Linear(in_features=100, out_features=50, bias=True)
  (dropout2): Dropout(p=0.5, inplace=False)
  (fc3): Linear(in_features=50, out_features=10, bias=True)
  (softmax): Softmax(dim=1)
)

In [ ]:
#훈련하기-1 epochs for문
num_epochs=50
for epoch in range(num_epochs):
    train_loss=train(model1, train_loader,criterion1,optimizer1,device)#loss만 return
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss:{train_loss:.4f}")

Epoch [1/50], Loss:1.7425
Epoch [2/50], Loss:1.6209
Epoch [3/50], Loss:1.5538
Epoch [4/50], Loss:1.5537
Epoch [5/50], Loss:1.5066
Epoch [6/50], Loss:1.6154
Epoch [7/50], Loss:1.5552
Epoch [8/50], Loss:1.5965
Epoch [9/50], Loss:1.4926
Epoch [10/50], Loss:1.6933
Epoch [11/50], Loss:1.5578
Epoch [12/50], Loss:1.5255
Epoch [13/50], Loss:1.4962
Epoch [14/50], Loss:1.5240
Epoch [15/50], Loss:1.5282
Epoch [16/50], Loss:1.5625
Epoch [17/50], Loss:1.5372
Epoch [18/50], Loss:1.5910
Epoch [19/50], Loss:1.4939
Epoch [20/50], Loss:1.6284
Epoch [21/50], Loss:1.5025
Epoch [22/50], Loss:1.4927
Epoch [23/50], Loss:1.5646
Epoch [24/50], Loss:1.5898
Epoch [25/50], Loss:1.5580
Epoch [26/50], Loss:1.5234
Epoch [27/50], Loss:1.5136
Epoch [28/50], Loss:1.5548
Epoch [29/50], Loss:1.7210
Epoch [30/50], Loss:1.6081
Epoch [31/50], Loss:1.4924
Epoch [32/50], Loss:1.5304
Epoch [33/50], Loss:1.5050
Epoch [34/50], Loss:1.5248
Epoch [35/50], Loss:1.5878
Epoch [36/50], Loss:1.5717
Epoch [37/50], Loss:1.6493
Epoch [38/

In [23]:
#훈련하기-2 epochs for문
num_epochs=50
for epoch in range(num_epochs):
    train_loss=train(model2, train_loader,criterion2,optimizer2,device)#loss만 return
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss:{train_loss:.4f}")

Epoch [1/50], Loss:1.6405
Epoch [2/50], Loss:1.6817
Epoch [3/50], Loss:1.6413
Epoch [4/50], Loss:1.6852
Epoch [5/50], Loss:1.5571
Epoch [6/50], Loss:1.5944
Epoch [7/50], Loss:1.6350
Epoch [8/50], Loss:1.6357
Epoch [9/50], Loss:1.7354
Epoch [10/50], Loss:1.5101
Epoch [11/50], Loss:1.7167
Epoch [12/50], Loss:1.6285
Epoch [13/50], Loss:1.6495
Epoch [14/50], Loss:1.5578
Epoch [15/50], Loss:1.6769
Epoch [16/50], Loss:1.6323
Epoch [17/50], Loss:1.5633
Epoch [18/50], Loss:1.5127
Epoch [19/50], Loss:1.6032
Epoch [20/50], Loss:1.6389
Epoch [21/50], Loss:1.5587
Epoch [22/50], Loss:1.6656
Epoch [23/50], Loss:1.5526
Epoch [24/50], Loss:1.5273
Epoch [25/50], Loss:1.6975
Epoch [26/50], Loss:1.6269
Epoch [27/50], Loss:1.5780
Epoch [28/50], Loss:1.5654
Epoch [29/50], Loss:1.7231
Epoch [30/50], Loss:1.5941
Epoch [31/50], Loss:1.6454
Epoch [32/50], Loss:1.6300
Epoch [33/50], Loss:1.5252
Epoch [34/50], Loss:1.6133
Epoch [35/50], Loss:1.5813
Epoch [36/50], Loss:1.5719
Epoch [37/50], Loss:1.5402
Epoch [38/

In [ ]:
#훈련하기-3 epochs for문
num_epochs=50
for epoch in range(num_epochs):
    train_loss=train(model3, train_loader,criterion3,optimizer3,device)#loss만 return
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss:{train_loss:.4f}")

Epoch [1/50], Loss:1.6894
Epoch [2/50], Loss:1.5921
Epoch [3/50], Loss:1.5095
Epoch [4/50], Loss:1.7331
Epoch [5/50], Loss:1.6518
Epoch [6/50], Loss:1.6239
Epoch [7/50], Loss:1.6175
Epoch [8/50], Loss:1.7344
Epoch [9/50], Loss:1.6359
Epoch [10/50], Loss:1.5846
Epoch [11/50], Loss:1.6873


In [ ]:
#훈련하기-4 epochs for문
num_epochs=50
for epoch in range(num_epochs):
    train_loss=train(model4, train_loader,criterion4,optimizer4,device)#loss만 return
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss:{train_loss:.4f}")

Epoch [1/50], Loss:1.6959
Epoch [2/50], Loss:1.5942
Epoch [3/50], Loss:1.6152
Epoch [4/50], Loss:1.6373
Epoch [5/50], Loss:1.8685
Epoch [6/50], Loss:1.7476
Epoch [7/50], Loss:1.5978
Epoch [8/50], Loss:1.6863
Epoch [9/50], Loss:1.8027
Epoch [10/50], Loss:1.6480
Epoch [11/50], Loss:1.6198
Epoch [12/50], Loss:1.5583
Epoch [13/50], Loss:1.6190
Epoch [14/50], Loss:1.6679
Epoch [15/50], Loss:1.5808
Epoch [16/50], Loss:1.5973
Epoch [17/50], Loss:1.5836
Epoch [18/50], Loss:1.6460
Epoch [19/50], Loss:1.5594
Epoch [20/50], Loss:1.5977
Epoch [21/50], Loss:1.6171
Epoch [22/50], Loss:1.6616
Epoch [23/50], Loss:1.5927
Epoch [24/50], Loss:1.5612
Epoch [25/50], Loss:1.5239
Epoch [26/50], Loss:1.6572
Epoch [27/50], Loss:1.6175
Epoch [28/50], Loss:1.5865
Epoch [29/50], Loss:1.6801
Epoch [30/50], Loss:1.6196
Epoch [31/50], Loss:1.5712
Epoch [32/50], Loss:1.5475
Epoch [33/50], Loss:1.6355
Epoch [34/50], Loss:1.6293
Epoch [35/50], Loss:1.5549
Epoch [36/50], Loss:1.6788
Epoch [37/50], Loss:1.5094
Epoch [38/

---
#Model1
:입력층->은닉층(활성화 함수)->출력층

In [ ]:
#훈련평가
train_loss, train_accuracy = evaluate(model1, train_loader, criterion1, device)
print(f"Train Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy:.4f}")

#검증평가
val_loss,val_accuracy=evaluate(model1,val_loader,criterion1,device)
print(f"Loss:{val_loss},Accuracy:{val_accuracy}")

Train Loss: 1.5447, Train Accuracy: 0.9167
Loss:1.5783970686594646,Accuracy:0.8828333333333334


In [ ]:
#일반화 평가
test_dataset=TensorDataset(test_scaled, test_target)
test_loader=DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

#평가하기
test_loss,test_accuracy=evaluate(model1,test_loader,criterion1,device)
print(f"Loss:{test_loss},Accuracy:{test_accuracy}")

Loss:1.5873503288902795,Accuracy:0.8736


In [ ]:
#Pandas로 정리하여 보기위해 list로 정리하기
model1Result=[train_accuracy,val_accuracy,test_accuracy]
model1Result

[0.9166666666666666, 0.8828333333333334, 0.8736]

---
#Model2
:입력층->은닉층(활성화 함수)->Dropout층->출력층

In [ ]:
#훈련평가
train_loss, train_accuracy = evaluate(model2, train_loader, criterion2, device)
print(f"Train Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy:.4f}")

#검증평가
val_loss,val_accuracy=evaluate(model2,val_loader,criterion2,device)
print(f"Loss:{val_loss},Accuracy:{val_accuracy}")

Train Loss: 1.5747, Train Accuracy: 0.8860
Loss:1.5937131551106771,Accuracy:0.8664166666666666


In [ ]:
#일반화 평가
test_dataset=TensorDataset(test_scaled, test_target)
test_loader=DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

#평가하기
test_loss,test_accuracy=evaluate(model2,test_loader,criterion2,device)
print(f"Loss:{test_loss},Accuracy:{test_accuracy}")

In [ ]:
#Pandas로 정리하여 보기위해 list로 정리하기
model2Result=[train_accuracy,val_accuracy,test_accuracy]
model2Result#떨어졌지만 달라지는것 확인함

In [ ]:
#Model3:입력층->은닉층(활성화 함수)->Dropout층->출력층
#훈련평가
train_loss, train_accuracy = evaluate(model3, train_loader, criterion3, device)
print(f"Train Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy:.4f}")

#검증평가
val_loss,val_accuracy=evaluate(model3,val_loader,criterion3,device)
print(f"Loss:{val_loss},Accuracy:{val_accuracy}")

#일반화 평가
test_dataset=TensorDataset(test_scaled, test_target)
test_loader=DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

#평가하기
test_loss,test_accuracy=evaluate(model3,test_loader,criterion3,device)
print(f"Loss:{test_loss},Accuracy:{test_accuracy}")

#Pandas로 정리하여 보기위해 list로 정리하기
model3Result=[train_accuracy,val_accuracy,test_accuracy]
model3Result#떨어졌지만 달라지는것 확인함

Train Loss: 1.5958, Train Accuracy: 0.8651
Loss:1.607670064608256,Accuracy:0.8530833333333333
Loss:1.6156391507139602,Accuracy:0.8454


[0.8651458333333333, 0.8530833333333333, 0.8454]

In [ ]:
#Model4:입력층->은닉층(활성화 함수)->Dropout층->출력층
#훈련평가
train_loss, train_accuracy = evaluate(model4, train_loader, criterion4, device)
print(f"Train Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy:.4f}")

#검증평가
val_loss,val_accuracy=evaluate(model4,val_loader,criterion4,device)
print(f"Loss:{val_loss},Accuracy:{val_accuracy}")

#일반화 평가
test_dataset=TensorDataset(test_scaled, test_target)
test_loader=DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

#평가하기
test_loss,test_accuracy=evaluate(model4,test_loader,criterion4,device)
print(f"Loss:{test_loss},Accuracy:{test_accuracy}")

#Pandas로 정리하여 보기위해 list로 정리하기
model4Result=[train_accuracy,val_accuracy,test_accuracy]
model4Result#떨어졌지만 달라지는것 확인함
#cnn:맥스플립

Train Loss: 1.5994, Train Accuracy: 0.8615
Loss:1.6084343334833782,Accuracy:0.8523333333333334
Loss:1.6154414506765982,Accuracy:0.8456


[0.8614791666666667, 0.8523333333333334, 0.8456]

In [ ]:
val_loss1, val_acc1 = evaluate(model1, val_loader, criterion1, device)
val_loss2, val_acc2 = evaluate(model2, val_loader, criterion2, device)
val_loss3, val_acc3 = evaluate(model3, val_loader, criterion3, device)
val_loss4, val_acc4 = evaluate(model4, val_loader, criterion4, device)

In [ ]:
import pandas as pd

result_df = pd.DataFrame({
    "Model": ["Model1", "Model2", "Model3", "Model4"],
    "Validation Loss": [val_loss1, val_loss2, val_loss3, val_loss4],
    "Validation Accuracy": [val_acc1, val_acc2, val_acc3, val_acc4]
})

result_df

,Model,Validation Loss,Validation Accuracy
0,Model1,1.578397,0.882833
1,Model2,1.593713,0.866417
2,Model3,1.607670,0.853083
3,Model4,1.608434,0.852333


In [ ]:
result_df = result_df.sort_values(by="Validation Accuracy", ascending=False)
result_df

,Model,Validation Loss,Validation Accuracy
0,Model1,1.578397,0.882833
1,Model2,1.593713,0.866417
2,Model3,1.607670,0.853083
3,Model4,1.608434,0.852333


In [ ]:
# result_df.style.format({
#     "Validation Loss": "{:.4f}",
#     "Validation Accuracy": "{:.4f}"
# })

In [ ]:
result_df = pd.DataFrame({
    "Model": ["Model1", "Model2", "Model3", "Model4"],
    "Val Loss": [val_loss1, val_loss2, val_loss3, val_loss4],
    "Val Accuracy": [val_acc1, val_acc2, val_acc3, val_acc4],
    # "Test Loss": [test_loss1, test_loss2, test_loss3, test_loss4],
    # "Test Accuracy": [test_acc1, test_acc2, test_acc3, test_acc4]
})

result_df

,Model,Val Loss,Val Accuracy
0,Model1,1.578397,0.882833
1,Model2,1.593713,0.866417
2,Model3,1.607670,0.853083
3,Model4,1.608434,0.852333
